# Shifts, Blending and Multicriteria Optimization — Sample Code
AIPI 530 · Fall 2026 · Dr. Natalia Summerville

Companion to the presentation, following the same format as. Run each problem from its “Load packages” cell onward. All five examples build an `AbstractModel` and supply data separately.

Uses the course environment with Pyomo, pandas, and GLPK. If GLPK is unavailable, install `highspy` and replace `SolverFactory('glpk')` with `SolverFactory('appsi_highs')`. These examples are linear or mixed-integer linear, so they use GLPK rather than HW3’s nonlinear solver IPOPT.

### Bean data files
Open this notebook from the **Code** folder. Keep the CSV files in the adjacent **data** folder:

- `../data/beans_ten.csv`: Problems 3, 4.

Columns: `Bean` is the unique bean identifier, `CostPerPound` is dollars per pound, and `FlavorPerPound` is flavor units per pound. Edit these files to change the bean inputs. Scenario settings such as quantity, flavor minimum, tolerance, and reward remain in the notebook. The numerical explanations describe the supplied CSV values.


# Problem 1 — Retail Associate Staffing


### Load packages


In [ ]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory


### Create data


In [ ]:
startData = pd.DataFrame({'Start': [10, 11, 12, 13, 14]})
demandData = pd.DataFrame({'Hour': range(10, 18),
                           'Demand': [2, 3, 7, 4, 4, 3, 5, 6]})
paramData = pd.DataFrame({'shiftLength': [4], 'shiftCost': [80]})

### Define model


In [ ]:
m = AbstractModel()
m.STARTS = Set()
m.HOURS = Set()
m.demand = Param(m.HOURS)
m.shiftLength = Param()
m.shiftCost = Param()

m.NStart = Var(m.STARTS, domain=NonNegativeIntegers)
m.NWorking = Var(m.HOURS, domain=NonNegativeReals)

def NWorking_Rule(m, wt):
    return m.NWorking[wt] == sum(m.NStart[st] for st in m.STARTS
                                if st <= wt < st + value(m.shiftLength))
m.Define_NWorking = Constraint(m.HOURS, rule=NWorking_Rule)

def Demand_Rule(m, wt):
    return m.NWorking[wt] >= m.demand[wt]
m.Cover_Demand = Constraint(m.HOURS, rule=Demand_Rule)

def Min_Payroll_Rule(m):
    return m.shiftCost * sum(m.NStart[st] for st in m.STARTS)
m.Min_Payroll = Objective(rule=Min_Payroll_Rule, sense=minimize)


### Create instance


In [ ]:
instanceData = {None: {
    'STARTS': {None: startData['Start'].tolist()},
    'HOURS': {None: demandData['Hour'].tolist()},
    'demand': demandData.set_index('Hour')['Demand'].to_dict(),
    'shiftLength': {None: paramData.loc[0, 'shiftLength']},
    'shiftCost': {None: paramData.loc[0, 'shiftCost']},
}}

instance = m.create_instance(instanceData)

### Solve


In [ ]:
solver = SolverFactory('glpk')
solution = solver.solve(instance)
check_optimal_termination(solution)
print(solution['Solver'])

### Print solution


In [ ]:

output = pd.DataFrame([
    {'Start hour': st, 'Associates starting': value(instance.NStart[st])}
    for st in instance.STARTS
])
print(output.round(4).to_string(index=False))
coverage = pd.DataFrame([
    {'Hour': wt, 'Required': value(instance.demand[wt]),
     'Working': value(instance.NWorking[wt])}
    for wt in instance.HOURS
])
print(coverage.round(4).to_string(index=False))
print(f'Total Payroll: ${value(instance.Min_Payroll):,.2f}')


### Explanation of results
The minimum payroll is **$1,040**, corresponding to 13 four-hour shifts at $80 each. One optimal schedule starts 2, 1, 4, 0, and 6 associates at hours 10, 11, 12, 13, and 14. Other optimal schedules may be returned.

At least seven associates must start by noon to cover noon demand, and six must start at 2 pm to cover the last hour. These disjoint groups require at least 13 associates.


# Problem 2 — Two-Bean Coffee Blending


### Load packages


In [ ]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

### Create data


In [ ]:
beanData = pd.DataFrame({'Bean': ['A', 'B'],
                         'CostPerPound': [6, 4],
                         'FlavorPerPound': [2, 1]})

paramData = pd.DataFrame({'quantity': [20], 'minimumFlavor': [30]})

### Define model


In [ ]:
m = AbstractModel()
m.BEANS = Set()
m.cost = Param(m.BEANS)
m.flavor = Param(m.BEANS)
m.quantity = Param()
m.minimumFlavor = Param()
m.x = Var(m.BEANS, domain=NonNegativeReals)
m.TotalCost = Var(domain=NonNegativeReals)
m.TotalFlavor = Var(domain=NonNegativeReals)

def TotalCost_Rule(m):
    return m.TotalCost == sum(m.cost[i] * m.x[i] for i in m.BEANS)
m.Define_TotalCost = Constraint(rule=TotalCost_Rule)

def TotalFlavor_Rule(m):
    return m.TotalFlavor == sum(m.flavor[i] * m.x[i] for i in m.BEANS)
m.Define_TotalFlavor = Constraint(rule=TotalFlavor_Rule)

def Quantity_Rule(m):
    return sum(m.x[i] for i in m.BEANS) == m.quantity
m.Blend_Quantity = Constraint(rule=Quantity_Rule)

def Minimum_Flavor_Rule(m):
    return m.TotalFlavor >= m.minimumFlavor
m.Minimum_Flavor = Constraint(rule=Minimum_Flavor_Rule)

def Min_Cost_Rule(m):
    return m.TotalCost
m.Min_Cost = Objective(rule=Min_Cost_Rule, sense=minimize)


### Create instance


In [ ]:
instanceData = {None: {
    'BEANS': {None: beanData['Bean'].tolist()},
    'cost': beanData.set_index('Bean')['CostPerPound'].to_dict(),
    'flavor': beanData.set_index('Bean')['FlavorPerPound'].to_dict(),
    'quantity': {None: paramData.loc[0, 'quantity']},
    'minimumFlavor': {None: paramData.loc[0, 'minimumFlavor']},
}}

instance = m.create_instance(instanceData)

### Solve


In [ ]:
solver = SolverFactory('glpk')
solution = solver.solve(instance)
assert check_optimal_termination(solution), 'Solver did not terminate optimally.'
print(solution['Solver'])

### Print solution


In [ ]:
print(solution['Solver'])
output = pd.DataFrame([
    {'Bean': i, 'Pounds': value(instance.x[i])}
    for i in instance.BEANS
])
print(output.round(4).to_string(index=False))
print(f'Total Cost: ${value(instance.TotalCost):,.2f}')
print(f'Total Flavor: {value(instance.TotalFlavor):.4f}')
print(f'Average Flavor: {value(instance.TotalFlavor) / value(instance.quantity):.4f}')


### Explanation of results
Use **10 lb of A and 10 lb of B**. The blend costs **$100**, contains 30 total flavor units, and averages 1.5 flavor units per pound. The flavor requirement binds: adding more premium Bean A would increase cost unnecessarily.


# Problem 3 — Ten-Bean Coffee Blending


### Create data


In [ ]:
beanData = pd.read_csv('../data/beans_ten.csv')

### Re-create instance

In [ ]:
instanceData = {None: {
    'BEANS': {None: beanData['Bean'].tolist()},
    'cost': beanData.set_index('Bean')['CostPerPound'].to_dict(),
    'flavor': beanData.set_index('Bean')['FlavorPerPound'].to_dict(),
    'quantity': {None: paramData.loc[0, 'quantity']},
    'minimumFlavor': {None: paramData.loc[0, 'minimumFlavor']},
}}

instance = m.create_instance(instanceData)

### Solve


In [ ]:
solver = SolverFactory('glpk')
solution = solver.solve(instance)

### Print solution


In [ ]:
print(solution['Solver'])
output = pd.DataFrame([
    {'Bean': i, 'Pounds': value(instance.x[i])}
    for i in instance.BEANS
])
print(output.round(4).to_string(index=False))
print(f'Total Cost: ${value(instance.TotalCost):,.2f}')
print(f'Total Flavor: {value(instance.TotalFlavor):.4f}')
print(f'Average Flavor: {value(instance.TotalFlavor) / value(instance.quantity):.4f}')


### Explanation of results
Use **15 lb of D and 5 lb of H**. The minimum cost is **$94**, total flavor is 30, and average flavor is 1.5. All other quantities are zero.

This standard blending problem keeps the minimum-flavor constraint. The next two multicriteria examples remove it. 


# Problem 4 — Sequential Multicriteria Blending


### Create data


In [ ]:
beanData = pd.read_csv('../data/beans_ten.csv')
paramData = pd.DataFrame({'quantity': [20], 'tolerance': [0.1]})